# AIML-02 — Spleen 3D U-Net (Colab T4)

MONAI 3D U-Net on **MSD Task 09 (Spleen)**. Trains from CT volumes with a foreground-focused patch sampler and evaluates with sliding-window Dice.

- Data: ~1.5 GB tar from the Medical Segmentation Decathlon.
- Runtime: ~30–60 min on a free T4 for 20 epochs.
- Output: `unet_model.pt` + `unet_metrics.json`.

> Education and research only. Not a medical device.


## 1. GPU check

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU and rerun.'


## 2. Install MONAI and fetch the data

In [ ]:
!pip install -q "monai>=1.4" nibabel
!wget -q --show-progress https://msd-for-monai.s3-us-west-2.amazonaws.com/Task09_Spleen.tar
!tar -xf Task09_Spleen.tar -C /content
!ls /content/Task09_Spleen | head
!ls /content/Task09_Spleen/imagesTr | head
!ls /content/Task09_Spleen/labelsTr | head


## 3. Clone the repo

In [ ]:
!rm -rf /content/nexvia
!git clone --depth 1 https://github.com/OmkarAla/nexvia.git /content/nexvia

## 4. Train

In [ ]:
!cd /content/nexvia && python AIML-02/training/train_unet.py   --data /content/Task09_Spleen   --out /content/unet-artifacts   --epochs 20   --run-id colab-unet-001

## 5. Inspect and download

In [ ]:
import json
m = json.load(open('/content/unet-artifacts/unet_metrics.json'))
print('best val Dice:', round(m['best_val_dice'], 4))
print('train/val volumes:', m['n_train'], m['n_val'], 'seconds:', m['train_seconds'])
!cd /content && zip -r unet-artifacts.zip unet-artifacts
from google.colab import files
files.download('/content/unet-artifacts.zip')

## Notes

- The default patch size is `(64, 64, 32)` with channels `(16, 32, 64, 128)`.
- Evaluation uses `sliding_window_inference` with 50% overlap.
- Place `unet_model.pt` in `AIML-02/artifacts/demo/` to enable CT segmentation in the worker.
